# Sesión 04 - Lab 2: Ingesta REST orquestada con Lakeflow Jobs

Este laboratorio construye un pipeline de ingesta REST de punta a punta contra una API pública real: la API de GitHub (`api.github.com`), consultando los repositorios públicos de la organización `databricks`. Sirve como reemplazo de "una API REST de un partner sin conector nativo" — el patrón de código (auth opcional, reintentos, aterrizaje, extracción incremental) es el mismo que correría contra cualquier partner real. Todo el pipeline usa el resultado real de la API, sin datos simulados: Lab 2A trae el extracto inicial, Lab 2C lo aterriza en Bronze, y Lab 2D vuelve a consultar la API para la extracción incremental. Requiere acceso saliente a internet (verificación de identidad con LinkedIn en Free Edition, o un workspace Premium). El Lab 2F crea un Lakeflow Job real (eso sí se puede hacer en Free Edition, sin acceso a internet de por medio) que orquesta el notebook.

## Verificación del entorno

In [0]:
dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion_04")

## Lab 2A: Cliente REST con autenticación y reintentos

Patrón genérico para consumir una API REST desde un notebook, probado en vivo contra la API pública de GitHub (`api.github.com`) — no hace falta una fuente propia como en el Lab 1, cualquier API pública sirve para ejercitar el patrón. Requiere acceso saliente a internet: en Free Edition alcanza con verificar tu identidad con LinkedIn (perfil de cuenta → *Verify with LinkedIn*); en un workspace Premium está disponible por defecto.

**Token opcional:** sin autenticación, GitHub limita a 60 requests/hora por IP — de sobra para esta demo. Con un personal access token (vía Secret Scope), el límite sube a 5000/hora — mismo patrón de credenciales que usaría la API de un partner real.

**Antipatrón:** guardar el token en la celda en vez de un Secret Scope — misma regla de credenciales hardcodeadas de la Sesión 01. **Distinguir** un `401`/`403` (credencial inválida, no reintentar) de un `429`/`5xx` (error transitorio, sí reintentar con backoff) — reintentar un error de autenticación no lo resuelve y satura la API sin necesidad. En una corrida en vivo normalmente vas a ver el intento 1 exitoso: el bucle de reintentos existe para el día que la API devuelva 429/5xx, algo difícil de forzar a demanda.

El resultado (`repos_databricks`) alimenta directo el resto del pipeline (Lab 2C en adelante) — nada río abajo depende de datos simulados.

In [0]:
import requests
import time

def obtener_token_opcional(scope, key):
    try:
        return dbutils.secrets.get(scope=scope, key=key)
    except Exception:
        return None

api_key = obtener_token_opcional("apis", "github_token")

# realizamos las peticiones
def llamar_api(url, headers, params=None, intentos=3):
    # realiza max 3 intentos
    for intento in range(intentos):
        respuesta = requests.get(url, headers=headers, params=params, timeout=30)
        if respuesta.status_code == 200:
            return respuesta.json()
        if respuesta.status_code == 429 or respuesta.status_code >= 500:
            time.sleep(2 ** intento)  # backoff exponencial
            continue
        respuesta.raise_for_status()  # 401/403/404: no tiene sentido reintentar
    raise RuntimeError(f"La API no respondió tras {intentos} intentos")

headers = {"Accept": "application/vnd.github+json"}
if api_key:
    headers["Authorization"] = f"Bearer {api_key}"

# obtenemos el listado de repositorios
repos_databricks = llamar_api(
    "https://api.github.com/orgs/databricks/repos",
    headers,
    params={"per_page": 10, "sort": "updated"},
)

print("Repos obtenidos:", len(repos_databricks))
for repo in repos_databricks:
    print("-", repo["name"], "|", repo["stargazers_count"], "estrellas")

## Lab 2B: Estrategias de aterrizaje del resultado

Una vez que el cliente REST (Lab 2A) trae la respuesta, hay dos formas de aterrizarla:

- **Directo a una tabla UC**: `spark.createDataFrame(registros)` sobre la lista de registros — más simple, suficiente para volúmenes bajos.
- **Crudo primero, a un Volume**: escribir la respuesta como archivo JSON antes de transformar — conserva el payload original para auditoría/reproceso, y permite después aplicar Auto Loader/`COPY INTO` sobre esos archivos igual que con cualquier otra fuente de archivos (Sesión 03).

Este lab usa el primer enfoque (directo a tabla), porque el volumen de repos por corrida es bajo — ver Lab 2C.

## Lab 2C: Aterrizar el resultado de Lab 2A en Bronze

`repos_databricks` — el resultado real que trajo Lab 2A — se aterriza directo, sin pasar por ningún archivo intermedio. Mismo criterio de esquema explícito y columnas de auditoría usado desde la Sesión 01.

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, StringType
from pyspark.sql.functions import col, current_timestamp, lit, to_timestamp

schema_repos = StructType([
    StructField("id", LongType(), False),
    StructField("name", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("description", StringType(), True),
    StructField("language", StringType(), True),
    StructField("stargazers_count", IntegerType(), True),
    StructField("forks_count", IntegerType(), True),
    StructField("open_issues_count", IntegerType(), True),
    StructField("updated_at", StringType(), True),
])

# creamos un dataframe con los datos de la petición
def proyectar_repos(repos):
    columnas = [campo.name for campo in schema_repos.fields]
    return [{c: repo.get(c) for c in columnas} for repo in repos]

df_repos = (
    spark.createDataFrame(proyectar_repos(repos_databricks), schema=schema_repos)
    .withColumn("updated_at", to_timestamp(col("updated_at")))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("github_api"))
    .withColumn("batch_id", lit("poll_" + datetime.now().strftime("%Y%m%d_%H%M")))
)

# Escribimos como tabla delta
df_repos.write.mode("overwrite").saveAsTable("dbassociate.default.repos_github_lab2")

print("Filas cargadas en el snapshot inicial:", df_repos.count())

## Lab 2D: Extracción incremental — segunda llamada real a la API

Vuelve a llamar al mismo endpoint y se queda solo con los repos cuyo `updated_at` es posterior al último valor ya cargado — mismo patrón que Lab 1D (JDBC), ahora aplicado a una llamada REST real en vez de un archivo estático.

Nota de diseño: puede traer 0 filas nuevas si nada cambió entre las dos corridas — es el comportamiento correcto de una extracción incremental idempotente, no un error.

In [0]:
ultima_fecha_procesada = spark.sql(
    "SELECT MAX(updated_at) AS ultima FROM dbassociate.default.repos_github_lab2"
).first()["ultima"]

print("Última fecha ya procesada en la tabla:", ultima_fecha_procesada)

# Nueva petición
respuesta_poll = llamar_api(
    "https://api.github.com/orgs/databricks/repos",
    headers,
    params={"per_page": 20, "sort": "updated"},
)

# tenemos todos los datos nuevos en base a la fecha de actualización
df_incremental = (
    spark.createDataFrame(proyectar_repos(respuesta_poll), schema=schema_repos)
    .withColumn("updated_at", to_timestamp(col("updated_at")))
    .filter(col("updated_at") > lit(ultima_fecha_procesada))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("github_api"))
    .withColumn("batch_id", lit("poll_" + datetime.now().strftime("%Y%m%d_%H%M")))
)

df_incremental.write.mode("append").saveAsTable("dbassociate.default.repos_github_lab2")

print("Filas agregadas en el poll incremental:", df_incremental.count())

## Lab 2E: Consulta de validación

In [0]:
spark.sql("""
    SELECT batch_id, COUNT(*) AS filas, MIN(updated_at) AS updated_at_min, MAX(updated_at) AS updated_at_max
    FROM dbassociate.default.repos_github_lab2
    GROUP BY batch_id
    ORDER BY batch_id
""").show(truncate=False)

## Lab 2F: Orquestar el notebook con un Lakeflow Job

A diferencia del acceso a internet, crear un Lakeflow Job **sí funciona en Free Edition** (hasta 5 tareas concurrentes por cuenta). Esto anticipa la Sesión 08/09 (Módulo 4), donde se cubren DAGs multi-tarea, reintentos y tipos de trigger en profundidad — hoy alcanza con un job de una sola tarea para cerrar el patrón "notebook que corre bien manualmente" → "pipeline programado".

**Pasos en la UI (Workflows → Jobs & Pipelines → Create Job):**

1. **Task name**: `ingesta_repos_github`.
2. **Type**: `Notebook`.
3. **Source**: `Workspace`, apuntando a este notebook (`sesion04_lab2.ipynb`).
4. **Compute**: Serverless.
5. **Trigger**: `Scheduled`, por ejemplo cada 15 minutos (frecuencia de polling razonable para monitorear actividad de repos) — el sílabo retoma triggers programados vs. basados en eventos en la Sesión 09.
6. **Create**.

No hace falta ejecutar este paso para completar el resto del notebook: es la parte de "orquestación" del taller, documentada acá para que se haga en vivo desde la UI del workspace.

## Limpieza

In [0]:
spark.sql("DROP TABLE IF EXISTS dbassociate.default.repos_github_lab2")

print("Tabla temporal de este laboratorio eliminada.")